# module-base-class-custom — worked example 1: __setattr__ registers params and submodules

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `module-base-class-custom`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

A minimal Module base class intercepts attribute assignment: when you do `self.weight = Parameter(...)` or `self.child = SomeModule()`, a custom `__setattr__` routes the value into `_parameters` or `_modules`. `parameters()` then walks both, recursing into submodules, so all trainable tensors are discoverable.

## Worked solution

We build the registrar and a recursive parameter walker.

1. **Bootstrap the registries.** In `__init__` we use `object.__setattr__` to install `_parameters` and `_modules` BEFORE the custom `__setattr__` can fire — otherwise it would try to read `_parameters` before it exists and recurse forever.
2. **Intercept assignment.** `__setattr__` checks the value's type: a `Parameter` goes into `_parameters[name]`, a `Module` into `_modules[name]`. We always also call `object.__setattr__` so the attribute is reachable as `self.name`.
3. **Recursive walk.** `parameters()` yields every direct Parameter, then `yield from m.parameters()` for each submodule — collecting the whole tree.

The demo builds a 2-layer model (each layer holds a weight Parameter) and prints that `parameters()` finds all of them.

In [ ]:
import numpy as np

class MiniTensor:
    def __init__(self, array, requires_grad=False):
        self.array = np.asarray(array, dtype=np.float64)
        self.requires_grad = requires_grad

class Parameter(MiniTensor):
    def __init__(self, array, requires_grad=True):
        super().__init__(array, requires_grad=requires_grad)

class Module:
    def __init__(self):
        object.__setattr__(self, '_parameters', {})
        object.__setattr__(self, '_modules', {})
    def __setattr__(self, name, value):
        if isinstance(value, Parameter):
            self._parameters[name] = value
            self._modules.pop(name, None)
        elif isinstance(value, Module):
            self._modules[name] = value
            self._parameters.pop(name, None)
        object.__setattr__(self, name, value)
    def parameters(self):
        for p in self._parameters.values():
            yield p
        for m in self._modules.values():
            yield from m.parameters()
    def forward(self, *a, **k):
        raise NotImplementedError()

class Layer(Module):
    def __init__(self, n):
        super().__init__()
        self.weight = Parameter(np.ones(n))

class Net(Module):
    def __init__(self):
        super().__init__()
        self.l1 = Layer(3)
        self.l2 = Layer(2)

net = Net()
params = list(net.parameters())
print('num params:', len(params))
print('all are Parameter:', all(isinstance(p, Parameter) for p in params))
print('total elements:', sum(p.array.size for p in params))